In [1]:
import pandas as pd
import os
from lsst.summit.utils import (
    ConsDbClient,
    getBandpassSeeingCorrection,
    getAirmassSeeingCorrection,
    makeDefaultButler
)
from sqlalchemy import create_engine
import numpy as np
from lsst.obs.lsst import LsstCam
from lsst.ts.ofc import BendModeToForce, OFCData, StateEstimator
from lsst.ts.wep.utils import convertZernikesToPsfWidth


        Use pytest instead. [astropy.tests.runner]


        Use pytest instead. [astropy.utils.decorators]


In [2]:
# Define ConsDbClient with specific URL
URL = "http://consdb-pq.consdb:8080/consdb"  
os.environ["no_proxy"] += ",.consdb"
cdb_client = ConsDbClient(URL)

ofc_data = OFCData("lsst")
ofc_data.controller["truncation_index"] = 6

m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:5] = True
m2_bending[:5] = True
ofc_data.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)
state_estimator = StateEstimator(ofc_data)

m2_bmf = BendModeToForce("M2", ofc_data)
m1m3_bmf = BendModeToForce("M1M3", ofc_data)

det_order = (191, 195, 199, 203)
camera = LsstCam().getCamera()
detector_names = [
    camera.get(det_id).getName() for det_id in det_order
]

In [3]:
query_table = """
        SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS exposure_id,
            q.visit_id AS visit_id,
            e.physical_filter as band,
            e.day_obs AS day,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.humidity AS humidity,
            e.pressure AS pressure,
            e.seq_num AS seq,
            e.sky_rotation AS sky_rotation,
            e.wind_dir AS wind_dir,
            e.wind_speed AS wind_speed,
            q.psf_sigma_median AS psf_fwhm,
            q.aos_fwhm,
            ccdvisit1_quicklook.z4,
            ccdvisit1_quicklook.z5,
            ccdvisit1_quicklook.z6,
            ccdvisit1_quicklook.z7,
            ccdvisit1_quicklook.z8,
            ccdvisit1_quicklook.z9,
            ccdvisit1_quicklook.z10,
            ccdvisit1_quicklook.z11,
            ccdvisit1_quicklook.z12,
            ccdvisit1_quicklook.z13,
            ccdvisit1_quicklook.z14,
            ccdvisit1_quicklook.z15,
            ccdvisit1_quicklook.z16,
            ccdvisit1_quicklook.z17,
            ccdvisit1_quicklook.z18,
            ccdvisit1_quicklook.z19,
            ccdvisit1_quicklook.z20,
            ccdvisit1_quicklook.z21,
            ccdvisit1_quicklook.z22,
            ccdvisit1_quicklook.z23,
            ccdvisit1_quicklook.z24,
            ccdvisit1_quicklook.z25,
            ccdvisit1_quicklook.z26,
            ccdvisit1_quicklook.z27,
            ccdvisit1_quicklook.z28,
            ccdvisit1.detector as detector,
            efd.mt_pointing_mount_position_rotator_mean AS rotation_angle,
            efd.mt_salindex110_sonic_temperature_mean AS sonic_temperature,
            efd.mt_salindex110_sonic_temperature_stddev_mean AS sonic_temperature_std,
            efd.mt_salindex1_temperature_0_mean AS cam_hex_temp_0,
            efd.mt_salindex1_temperature_1_mean AS cam_hex_temp_1,
            efd.mt_salindex1_temperature_2_mean AS cam_hex_temp_2,
            efd.mt_salindex1_temperature_3_mean AS cam_hex_temp_3,
            efd.mt_salindex1_temperature_4_mean AS cam_hex_temp_4,
            efd.mt_salindex1_temperature_5_mean AS cam_hex_temp_5,
            efd.mt_salindex1_temperature_6_mean AS cam_hex_temp_6,
            efd.mt_salindex1_temperature_7_mean AS cam_hex_temp_7,
            efd.mt_salindex301_temperature_0_mean AS outside_temp_0,
            efd.mt_salindex301_temperature_1_mean AS outside_temp_1,
            efd.mt_salindex301_temperature_2_mean AS outside_temp_2,
            efd.mt_salindex301_temperature_3_mean AS outside_temp_3,
            efd.mt_salindex301_temperature_4_mean AS outside_temp_4,
            efd.mt_salindex301_temperature_5_mean AS outside_temp_5,
            efd.mt_salindex301_temperature_6_mean AS outside_temp_6,
            efd.mt_salindex301_temperature_7_mean AS outside_temp_7
        FROM 
            cdb_lsstcam.exposure AS e,
            cdb_lsstcam.visit1_quicklook AS q,
            efd_lsstcam.exposure_efd AS efd,
            cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1
        WHERE 
            ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
            AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND ccdvisit1.visit_id = efd.exposure_id
            AND (e.img_type = 'science' or e.img_type = 'acq')
            AND e.day_obs > 20250528
            AND e.airmass > 0
            AND e.band != 'none'
        
        --order-by e.seq_num
"""
result_df = cdb_client.query(query_table).to_pandas()
zernike_columns = [f"z{i}" for i in range(4, 29)]
result_df["zernikes"] = result_df[zernike_columns].apply(
    lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
)

/tmp/ipykernel_6040/1737501411.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1


In [4]:
result_df.columns

Index(['air_temp', 'airmass', 'dimm', 'elevation', 'azimuth', 'exposure_id',
       'visit_id', 'band', 'day', 'time', 'seeing', 'humidity', 'pressure',
       'seq', 'sky_rotation', 'wind_dir', 'wind_speed', 'psf_fwhm', 'aos_fwhm',
       'z4', 'z5', 'z6', 'z7', 'z8', 'z9', 'z10', 'z11', 'z12', 'z13', 'z14',
       'z15', 'z16', 'z17', 'z18', 'z19', 'z20', 'z21', 'z22', 'z23', 'z24',
       'z25', 'z26', 'z27', 'z28', 'detector', 'rotation_angle',
       'sonic_temperature', 'sonic_temperature_std', 'cam_hex_temp_0',
       'cam_hex_temp_1', 'cam_hex_temp_2', 'cam_hex_temp_3', 'cam_hex_temp_4',
       'cam_hex_temp_5', 'cam_hex_temp_6', 'cam_hex_temp_7', 'outside_temp_0',
       'outside_temp_1', 'outside_temp_2', 'outside_temp_3', 'outside_temp_4',
       'outside_temp_5', 'outside_temp_6', 'outside_temp_7', 'zernikes'],
      dtype='object')

In [5]:
# At this point we have queried the classic consdb which is also available at the summit
# now let's dig through the most interesting one, the efd_lsstcam
# you may consider adding more properties, here is a few that I am interested in
# Query unpivoted data based on the previous result 
exposure_ids = ', '.join(map(str, result_df["exposure_id"].values))

query_unpivoted = f"""
SELECT
    exposure_id,
    property,
    field,
    value
FROM
    efd_lsstcam.exposure_efd_unpivoted
WHERE
    exposure_id IN ({exposure_ids})
    AND (property = 'mt_m2_axial_force_lut_gravity_mean'
        OR property = 'mt_m2_axial_force_lut_temperature_mean'
        OR property = 'mt_m1m3_applied_elevation_forces_mean'
        OR property = 'mt_logevent_aggregated_dof'
        OR property = 'mt_pointing_mount_position_rotator_mean'
    );
"""

rename_dict = {'mt_m2_axial_force_lut_gravity_mean':'m2',
               'mt_m2_axial_force_lut_temperature_mean':'m2',
               'mt_m1m3_applied_azimuth_forces_mean': 'azimuth',
               'mt_m1m3_applied_thermal_forces_mean': 'thermal_forces',
               'mt_m1m3_applied_elevation_forces_mean': 'elevation',
               'mt_logevent_aggregated_dof':'',
              }
result_unp_df = cdb_client.query(query_unpivoted).to_pandas()

result_unp_df['property'] = result_unp_df['property'].replace(rename_dict)

result_unp_df['property_field'] = result_unp_df['property'] + '_' + result_unp_df['field']

result_unp_df = result_unp_df.pivot(columns='property_field', values='value', index='exposure_id').reset_index()

# Merge tabular and unpivoted dataframes
result = pd.merge(result_df, result_unp_df, on='exposure_id', how='inner')

gravity_cols = [f"m2_lutGravity{i}" for i in range(72)]
temp_cols = [f"m2_lutTemperature{i}" for i in range(72)]

result["m2_lut_gravity"] = result[gravity_cols].values.tolist()
result["m2_lut_temp"] = result[temp_cols].values.tolist()
gravity_array = result[gravity_cols].values.astype(float)
temp_array = result[temp_cols].values.astype(float)

# Element-wise sum across each row
result["m2_lut"] = (gravity_array + temp_array).tolist()
result["m2_bmf"] = result["m2_lut"].apply(lambda x: m2_bmf.bending_mode(np.array(x)))

result.drop(columns=gravity_cols, inplace=True)
result.drop(columns=temp_cols, inplace=True)

force_types = ['elevation']
cols_list = [[f"{ft}_zForces{i}" for i in range(156)] for ft in force_types]
all_cols = [col for cols in cols_list for col in cols]

result["m1m3_lut"] = (result[cols_list[0]].values).tolist()
result["m1m3_bmf"] = result["m1m3_lut"].apply(lambda x: m1m3_bmf.bending_mode(np.array(x)))
result.drop(columns=all_cols, inplace=True)

aggregated_dof_cols = [f"_aggregatedDoF{i}" for i in range(50)]
values = result[aggregated_dof_cols].values.tolist()
result["dof_state"] = result[aggregated_dof_cols].values.tolist()
result.drop(columns=aggregated_dof_cols, inplace=True)

temp_cols = [f"cam_hex_temp_{i}" for i in range(8)]
result["cam_hex_temp_avg"] = result[temp_cols].mean(axis=1)
result.drop(columns=temp_cols, inplace=True)

outside_temp_cols = [f"outside_temp_{i}" for i in range(8)]
result["outside_temp_avg"] = result[outside_temp_cols].mean(axis=1)
result.drop(columns=outside_temp_cols, inplace=True)

result

,air_temp,airmass,dimm,elevation,azimuth,exposure_id,visit_id,band,day,time,...,zernikes,m2_lut_gravity,m2_lut_temp,m2_lut,m2_bmf,m1m3_lut,m1m3_bmf,dof_state,cam_hex_temp_avg,outside_temp_avg
0,9.350,1.068967,0.733674,69.321770,44.801426,2025060800527,2025060800527,i_39,20250608,2025-06-09T08:05:34.266000,...,"[0.27354729175567627, 0.2948545399155138, -0.6...","[132.34775806451614, 224.24459677419355, 157.8...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[132.34775806451614, 224.24459677419355, 157.8...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[500.25123023236756, 0.006612184305930973, -0....",10.743179,9.359375
1,9.325,1.042343,0.758396,73.617141,25.061065,2025060800537,2025060800537,i_39,20250608,2025-06-09T08:13:10.814000,...,"[-0.10942893475294113, -1.0901161767313288, -0...","[142.94887096774193, 228.0233548387097, 163.47...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[142.94887096774193, 228.0233548387097, 163.47...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[723.072990631177, 813.7046534812379, 957.4344...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[567.2274734513863, 0.006603410112026583, -0.0...",10.757381,9.365625
2,10.150,1.265369,None,52.200842,58.206741,2025070100603,2025070100603,i_39,20250701,2025-07-02T06:20:39.283000,...,"[-0.7671253681182861, -0.2528946001488741, 0.3...","[nan, nan, 124.57422580645161, 282.75882258064...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",12.025941,10.1625
3,9.200,1.156099,None,59.895530,87.021343,2025062300412,2025062300412,i_39,20250623,2025-06-24T03:04:20,...,"[0.8182212114334106, 0.7392233295862685, -0.10...","[103.3909677419355, 211.50275806451612, 141.71...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[103.3909677419355, 211.50275806451612, 141.71...","[-1.0731143140755242, 13.019119727278113, -1.7...","[651.451588769996, 732.3572746044142, 862.7580...","[0.036080270818627636, 0.25859081148872687, -1...","[1458.7313262375374, -69.4529895654239, 567.35...",11.625719,9.08125
4,6.825,1.152615,None,60.056173,244.191030,2025063000494,2025063000494,r_57,20250630,2025-07-01T06:37:14.655000,...,"[0.028768934309482574, 0.4593045196203513, -0....","[nan, nan, nan, nan, nan, nan, nan, nan, 208.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[nan, nan, nan, nan, nan, nan, nan, nan, 208.4...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",11.852029,6.817187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26900,7.450,1.522570,1.641455,41.022340,95.330760,2025070600190,2025070600190,y_10,20250706,2025-07-07T01:16:47.764000,...,"[-0.46218016743659973, 0.06404385413641467, -0...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[493.82469817917666, 554.2043403448357, 654.37...","[0.07382186157641168, 0.22242970819940666, -1....","[1611.246999620807, -69.55740341890196, 567.23...",12.743331,7.490625
26901,7.500,1.395953,1.205963,45.734364,100.371524,2025070600193,2025070600193,y_10,20250706,2025-07-07T01:18:45.034000,...,"[-0.7096647620201111, -0.2515487651053806, 0.3...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, 

In [6]:
import pandas as pd
import os
from lsst.summit.utils import (
    ConsDbClient,
    getBandpassSeeingCorrection,
    getAirmassSeeingCorrection,
    makeDefaultButler
)

# Define ConsDbClient with specific URL
URL = "http://consdb-pq.consdb:8080/consdb"  
os.environ["no_proxy"] += ",.consdb"
cdb_client = ConsDbClient(URL)

# Query tabular data from ConsDbClient and convert to DataFrame
query_table = """
        SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS exposure_id,
            e.physical_filter as band,
            e.day_obs AS day,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.humidity AS humidity,
            e.pressure AS pressure,
            e.seq_num AS seq,
            e.sky_rotation AS sky_rotation,
            e.wind_dir AS wind_dir,
            e.wind_speed AS wind_speed,
            q.sky_noise_median AS sky_noise,
            q.sky_noise_max AS sky_noise_max,
            q.sky_noise_min AS sky_noise_min,
            q.sky_bg_median AS sky_bg,
            q.sky_bg_max AS sky_bg_max,
            q.sky_bg_min AS sky_bg_min,
            q.psf_sigma_median AS psf_fwhm,
            q.psf_sigma_min AS psf_fwhm_min,
            q.psf_sigma_max AS psf_fwhm_max,
            q.psf_ixx_median AS psf_ixx_median,
            q.psf_ixx_max AS psf_ixx_max,
            q.psf_ixx_min AS psf_ixx_min,
            q.psf_iyy_median AS psf_iyy_median,
            q.psf_iyy_max AS psf_iyy_max,
            q.psf_iyy_min AS psf_iyy_min,
            q.psf_ixy_median AS psf_ixy_median,
            q.psf_ixy_max AS psf_ixy_max,
            q.psf_ixy_min AS psf_ixy_min,
            q.psf_area_max AS psf_area_max,
            q.psf_area_median AS psf_area_median,
            q.psf_area_min AS psf_area_min
        FROM 
            cdb_lsstcam.exposure AS e
        JOIN 
            cdb_lsstcam.visit1_quicklook AS q
            ON e.exposure_id = q.visit_id
        WHERE 
            (e.img_type = 'OBJECT' or e.img_type = 'ACQ') AND
            e.day_obs > 20250501 AND
            e.airmass > 0 AND
            e.band != 'none'
        
        --order-by e.seq_num
"""
result_df = cdb_client.query(query_table).to_pandas()

# Query unpivoted data based on the previous result 
exposure_ids = ', '.join(map(str, result_df["exposure_id"].values))
query_unpivoted = f"""
SELECT
    exposure_id,
    property,
    field,
    value
FROM
    efd_lsstcam.exposure_efd_unpivoted
WHERE
    exposure_id IN ({exposure_ids})
    AND property = 'mt_m2_axial_force_lut_temperature_mean'
"""


result_unp_df = cdb_client.query(query_unpivoted).to_pandas()
print(result_unp_df)
result_unp_df['property_field'] = result_unp_df['property'] + '_' + result_unp_df['field']

result_unp_df = result_unp_df.pivot(columns='property_field', values='value', index='exposure_id').reset_index()

# Merge tabular and unpivoted dataframes
result = pd.merge(result_df, result_unp_df, on='exposure_id', how='inner')

HTTPError: 500 Server Error: Internal Server Error for url: http://@consdb-pq.consdb/consdb/query